# Multi-modal RAG: text plus image embeddings

A product team is building a visual product search bot for an e-commerce catalog. Customers paste an image of a chair they saw in a hotel lobby, type "find me something like this but cheaper, in oak," and expect the bot to return both text descriptions of matching products AND visually similar product images. You have 75 minutes to build a multimodal RAG pipeline that indexes a 200-item catalog (100 text + 100 image) into a unified Pinecone Serverless index using Cohere Embed v3 multimodal embeddings, runs cross-modal queries (image-to-text and text-to-image), and grounds Claude Sonnet 4.6 generation in retrieved chunks from both modalities.

The killer moment: a query like `'a wooden chair'` returns five image vectors, you base64-encode the top-3 and feed them to Claude as `image` content blocks alongside the text prompt, and the answer cites both `[product_<id>]` rows from text descriptions AND `[product_<id>]` rows from images.

Estimated time: 75 minutes (session window 105). Cost: about $1.50 to $2.50 on a clean run; up to $3.50 if you re-run the 10-query grounded eval while debugging. Claude Sonnet with image input is the main spend (each image you pass counts as roughly 1100 tokens, so a generation call with 3 retrieved images costs more than a text-only generation). Cohere embeddings and Pinecone Serverless are pennies or free tier.

Two bucks. Cross-modal recall is the moment that earns it.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. labrun-checks stays on 0.7.0 per the track. cohere is
# pinned to the v5 line which exposes the multimodal Embed v3 surface;
# pinecone-client 5.x is the Serverless-native release; anthropic 0.42.0
# carries the image-content-block API; Pillow draws the synthetic PNGs.

!pip install --quiet labrun-checks==0.7.0 cohere==5.11.4 pinecone-client==5.0.1 anthropic==0.42.0 pillow==10.4.0

In [ ]:
# Imports and per-lab constants. The Pinecone index name uses the lab slug
# verbatim; the orphan scan matches by `labrun-` prefix across list_indexes.

import atexit
import base64
import getpass
import io
import json
import os
import random
import re
import shutil
import sys
import time
from pathlib import Path

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "multimodal-rag-text-image"
LAB_TAG_VALUE = LAB_ID

PINECONE_INDEX_NAME = "labrun-multimodal-rag-text-image-products"
PINECONE_PREFIX = "labrun-"
IMAGE_DIR = "/content/product_images"
TRACE_PATH = "/content/multimodal_traces.json"

COHERE_MULTIMODAL_MODEL = "embed-v3.0-multimodal"
COHERE_EMBEDDING_DIM = 1024
ANTHROPIC_MODEL = "claude-sonnet-4-6"
ANTHROPIC_HAIKU_MODEL = "claude-haiku-4-5"

# Deterministic catalog: 100 text-only products (descriptions) plus 100 image
# products (synthetic PNG placeholders). Seed locks the colors and shapes so
# checkpoints are reproducible across re-runs.
CATALOG_SEED = 4242
CATALOG_TEXT_COUNT = 100
CATALOG_IMAGE_COUNT = 100
TOTAL_VECTORS = CATALOG_TEXT_COUNT + CATALOG_IMAGE_COUNT

In [ ]:
# NBVAL_SKIP
# BYOK setup: session token, Cohere, Pinecone, Anthropic. Voyage is an
# optional fallback; if VOYAGE_API_KEY is blank we skip the Voyage branch.
# Ping each provider cheaply to fail fast on 401, then register() the
# labrun session so the companion panel sees this run.

import cohere
from pinecone import Pinecone, ServerlessSpec
import anthropic

session_token = getpass.getpass("Paste your session token from labrun.dev: ").strip()
COHERE_API_KEY = getpass.getpass("COHERE_API_KEY: ").strip()
PINECONE_API_KEY = getpass.getpass("PINECONE_API_KEY (leave blank for Qdrant fallback): ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
VOYAGE_API_KEY = getpass.getpass("VOYAGE_API_KEY (optional, press Enter to skip): ").strip()

QDRANT_API_KEY = ""
QDRANT_URL = ""
if not PINECONE_API_KEY:
    QDRANT_API_KEY = getpass.getpass("QDRANT_API_KEY (Qdrant Cloud fallback): ").strip()
    QDRANT_URL = getpass.getpass("QDRANT_URL (https://xxx.qdrant.cloud): ").strip()
    print("Pinecone key blank. Qdrant fallback path is not the supported v1 path for this lab.")
    print("This notebook validates the Pinecone path; re-run with PINECONE_API_KEY set.")
    raise SystemExit(1)

if not all([session_token, COHERE_API_KEY, PINECONE_API_KEY, ANTHROPIC_API_KEY]):
    print("session_token, COHERE_API_KEY, PINECONE_API_KEY, ANTHROPIC_API_KEY are all required.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["COHERE_API_KEY"] = COHERE_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

# Cohere ping. models.list() is free; refuse on 401.
try:
    co = cohere.Client(api_key=COHERE_API_KEY)
    _ = co.models.list()
    print("Cohere credentials validated.")
except Exception as exc:
    print(f"Cohere key rejected: {exc!r}")
    print("Refresh COHERE_API_KEY and re-run this cell.")
    raise SystemExit(1)

# Pinecone ping. list_indexes is free; refuse on 401.
try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
    _existing = pc.list_indexes()
    print("Pinecone credentials validated.")
except Exception as exc:
    print(f"Pinecone key rejected: {exc!r}")
    print("Refresh PINECONE_API_KEY and re-run this cell.")
    raise SystemExit(1)

# Anthropic ping. One Haiku call: a fraction of a cent.
try:
    _anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = _anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    print("Refresh ANTHROPIC_API_KEY and re-run this cell.")
    raise SystemExit(1)

SESSION_CREDENTIALS = {
    "cohere_api_key": COHERE_API_KEY,
    "pinecone_api_key": PINECONE_API_KEY,
    "anthropic_api_key": ANTHROPIC_API_KEY,
    "voyage_api_key": VOYAGE_API_KEY,
    "qdrant_api_key": QDRANT_API_KEY,
    "qdrant_url": QDRANT_URL,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=SESSION_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom Pinecone adapter, atexit safety net, orphan scan.
# Manifest is module-level. Adapter deletes the Pinecone index by name and
# removes the local image dir and trace file.
# TODO: migrate to vector_db.py once the Pinecone driver ships.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="pinecone_index",
        id=PINECONE_INDEX_NAME,
        region="pinecone",
        cli_delete_command=f"pinecone index delete {PINECONE_INDEX_NAME}",
    ),
    CleanupResource(
        type="local_dir",
        id=IMAGE_DIR,
        region="local",
        cli_delete_command=f"rm -rf {IMAGE_DIR}",
    ),
    CleanupResource(
        type="local_file",
        id=TRACE_PATH,
        region="local",
        cli_delete_command=f"rm -f {TRACE_PATH}",
    ),
]


class MultimodalRagCleanupAdapter:
    """Deletes the Pinecone index by name, removes the local image dir, and
    removes the local trace file. Implements the CleanupAdapter protocol.

    TODO: migrate to vector_db.py once Pinecone driver ships. For now the
    adapter lives inline so the lab is self-contained.
    """

    def __init__(self, pinecone_client):
        self._pc = pinecone_client

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "pinecone_index":
            try:
                existing = [ix["name"] for ix in self._pc.list_indexes()]
                if rid in existing:
                    self._pc.delete_index(rid)
            except Exception as exc:
                msg = repr(exc).lower()
                if "not found" in msg or "404" in msg or "does not exist" in msg:
                    return
                raise
        elif rtype == "local_dir":
            shutil.rmtree(rid, ignore_errors=True)
        elif rtype == "local_file":
            try:
                os.remove(rid)
            except FileNotFoundError:
                pass
        else:
            raise ValueError(f"MultimodalRagCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "pinecone_index":
            try:
                existing = [ix["name"] for ix in self._pc.list_indexes()]
                return rid in existing
            except Exception:
                return False
        if rtype == "local_dir":
            return os.path.isdir(rid)
        if rtype == "local_file":
            return os.path.exists(rid)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # Pinecone Serverless indexes do not carry user-facing tags. The
        # orphan scan iterates list_indexes() and returns any name that
        # starts with the labrun- prefix so leftover indexes from a previous
        # attempt surface here.
        try:
            existing = [ix["name"] for ix in self._pc.list_indexes()]
            return [name for name in existing if name.startswith(PINECONE_PREFIX)]
        except Exception:
            return []


CLEANUP_ADAPTER = MultimodalRagCleanupAdapter(pinecone_client=pc)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(SESSION_CREDENTIALS, LAB_TAG_VALUE, "pinecone")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing Pinecone indexes with prefix {PINECONE_PREFIX!r} were found:")
    for name in orphans:
        print("  -", name)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

In [ ]:
# NBVAL_SKIP
# Seed the 200-item synthetic product catalog. 100 text-only descriptions and
# 100 image products. Both share product_id space [0, 199]. The image half
# is generated as small PNG placeholders via PIL: colored rectangles with the
# product label rendered, ~6 KB each, deterministic by CATALOG_SEED.
#
# Ground truth: each test query is hand-labelled with a list of relevant
# product_ids spanning both modalities so cross-modal retrieval can be
# scored. The eval JSON lives in this cell as RELEVANCE_LABELS.

from PIL import Image, ImageDraw, ImageFont

random.seed(CATALOG_SEED)

_CATEGORIES = [
    ("chair", "wooden"), ("chair", "oak"), ("chair", "velvet"),
    ("lamp", "brass"), ("lamp", "copper"), ("lamp", "ceramic"),
    ("vase", "glass"), ("vase", "clay"), ("vase", "porcelain"),
    ("desk", "oak"), ("desk", "walnut"), ("desk", "steel"),
    ("rug", "wool"), ("rug", "jute"), ("rug", "silk"),
    ("sofa", "linen"), ("sofa", "leather"),
    ("bookshelf", "pine"), ("bookshelf", "steel"),
    ("mirror", "gold-framed"),
]

_COLORS = [
    (160, 110, 60),   # oak
    (110, 70, 40),    # walnut
    (200, 50, 50),    # velvet red
    (220, 180, 80),   # brass
    (180, 100, 60),   # copper
    (240, 240, 220),  # ceramic
    (60, 130, 180),   # glass blue
    (200, 200, 200),  # steel
    (90, 60, 30),     # leather
    (230, 220, 200),  # linen
]

Path(IMAGE_DIR).mkdir(parents=True, exist_ok=True)

CATALOG = []  # list of {product_id, modality, description?, image_path?, category, material}

# Text-only products: ids 0..99
for pid in range(CATALOG_TEXT_COUNT):
    category, material = random.choice(_CATEGORIES)
    descriptor = random.choice(["compact", "sculptural", "minimal", "industrial", "vintage", "modern"])
    price = random.choice([89, 129, 199, 259, 349, 499, 699])
    description = (
        f"A {descriptor} {material} {category} priced at ${price}. "
        f"Suits a {random.choice(['hotel lobby', 'reading nook', 'studio apartment', 'dining room', 'home office'])}. "
        f"Care: {random.choice(['wipe with dry cloth', 'avoid direct sun', 'professional clean only'])}."
    )
    CATALOG.append({
        "product_id": pid,
        "modality": "text",
        "description": description,
        "category": category,
        "material": material,
    })

# Image products: ids 100..199. Each PNG is a small colored rectangle with
# the category label rendered. This is a placeholder for real product
# photography; the cross-modal recall numbers reflect that. The point of
# the lab is the API shape and the unified-index trade-off, not photoreal.
def _draw_product_png(category, material, color, path):
    img = Image.new("RGB", (256, 256), color)
    draw = ImageDraw.Draw(img)
    # Border
    draw.rectangle([(8, 8), (247, 247)], outline=(20, 20, 20), width=4)
    # Inner shape varies by category
    cx, cy = 128, 128
    if category in ("chair", "sofa", "desk", "bookshelf"):
        draw.rectangle([(cx - 60, cy - 30), (cx + 60, cy + 60)], fill=(40, 40, 40))
    elif category in ("lamp", "vase"):
        draw.ellipse([(cx - 40, cy - 60), (cx + 40, cy + 60)], fill=(40, 40, 40))
    elif category == "rug":
        draw.rectangle([(cx - 80, cy - 20), (cx + 80, cy + 20)], fill=(40, 40, 40))
    elif category == "mirror":
        draw.ellipse([(cx - 50, cy - 70), (cx + 50, cy + 70)], outline=(40, 40, 40), width=8)
    draw.text((16, 220), f"{material} {category}", fill=(20, 20, 20))
    img.save(path, format="PNG", optimize=True)


for offset in range(CATALOG_IMAGE_COUNT):
    pid = CATALOG_TEXT_COUNT + offset
    category, material = random.choice(_CATEGORIES)
    color = random.choice(_COLORS)
    image_path = os.path.join(IMAGE_DIR, f"product_{pid}.png")
    _draw_product_png(category, material, color, image_path)
    CATALOG.append({
        "product_id": pid,
        "modality": "image",
        "image_path": image_path,
        "category": category,
        "material": material,
    })

print(f"Catalog ready: {len(CATALOG)} items ({CATALOG_TEXT_COUNT} text, {CATALOG_IMAGE_COUNT} image).")
print(f"Images written to {IMAGE_DIR}.")

# Test queries with ground-truth relevant product_ids spanning both modalities.
# Five image-input queries (the lab supplies the image bytes from its own
# catalog ids 100, 110, 120, 130, 140) and five text-input queries.
TEST_IMAGE_QUERY_IDS = [100, 110, 120, 130, 140]
TEST_TEXT_QUERIES = [
    "a wooden chair for a hotel lobby",
    "a brass lamp on a writing desk",
    "a glass vase for the dining room",
    "an oak desk for a home office",
    "a wool rug for a reading nook",
]

# Build relevance labels: for each query, the relevant products are every
# catalog item that matches the query's category, spanning both modalities.
RELEVANCE_LABELS = {"image_queries": {}, "text_queries": {}}

for qid in TEST_IMAGE_QUERY_IDS:
    qitem = CATALOG[qid]
    relevant = [c["product_id"] for c in CATALOG if c["category"] == qitem["category"] and c["product_id"] != qid]
    RELEVANCE_LABELS["image_queries"][str(qid)] = relevant

_TEXT_QUERY_CATEGORIES = ["chair", "lamp", "vase", "desk", "rug"]
for q, cat in zip(TEST_TEXT_QUERIES, _TEXT_QUERY_CATEGORIES):
    relevant = [c["product_id"] for c in CATALOG if c["category"] == cat]
    RELEVANCE_LABELS["text_queries"][q] = relevant

print("Relevance labels assembled for 5 image queries and 5 text queries.")

## Task 1: Build the unified Pinecone index

Create a Pinecone Serverless index named `labrun-multimodal-rag-text-image-products` with cosine similarity and 1024 dimensions (Cohere Embed v3 multimodal). Then embed each catalog item with Cohere `embed-v3.0-multimodal`: text products via `input_type='search_document'` on the description, image products via the `inputs=[{'image': base64_str}]` form. Upsert all 200 vectors with metadata `{'modality': 'text' | 'image', 'product_id': ..., 'category': ..., 'material': ...}`.

Two designs the brief calls out:

1. **One model, both modalities.** `embed-v3.0` (text-only) and `embed-v3.0-multimodal` return vectors of the same dimension but live in different spaces. Mixing them is the single most common bug in this lab. Use the multimodal variant for both halves.
2. **Metadata carries modality.** Every upsert sets `modality` so the validator can sample 10 random vectors and confirm both values are present. Skipping metadata is the second most common bug.

In [ ]:
# NBVAL_SKIP
# Task 1: build the unified Pinecone index and upsert 200 vectors.
#
# Level 2 skeleton: the index-creation call, the Cohere embedding calls for
# both modalities, and the upsert with metadata are yours to write. The
# rest of the cell (image base64 helper, batching loop scaffold, polling
# loop for index readiness, print statements) is provided.

def _png_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")


# Create the Pinecone Serverless index. cosine similarity. 1024 dims.
# YOUR CODE: call pc.create_index(name=PINECONE_INDEX_NAME, dimension=COHERE_EMBEDDING_DIM,
#            metric='cosine', spec=ServerlessSpec(cloud='aws', region='us-east-1'))
#            then poll pc.describe_index(PINECONE_INDEX_NAME).status['ready']
#            until it returns True.

# (Replace this block with the create_index call and the readiness poll.)

index = pc.Index(PINECONE_INDEX_NAME)

# Embed and upsert the 100 text products.
TEXT_BATCH = 32
text_items = [c for c in CATALOG if c["modality"] == "text"]
for batch_start in range(0, len(text_items), TEXT_BATCH):
    batch = text_items[batch_start:batch_start + TEXT_BATCH]
    texts = [c["description"] for c in batch]

    # YOUR CODE: call co.embed(texts=texts, model=COHERE_MULTIMODAL_MODEL,
    #            input_type='search_document') and pull .embeddings (a list of
    #            1024-dim float lists). Store as `text_vectors`.
    text_vectors = []  # replace with the embed call result

    # YOUR CODE: build upsert items as a list of tuples (id, vector, metadata)
    #            where id = f"product_{c['product_id']}", vector = text_vectors[i],
    #            metadata = {'modality': 'text', 'product_id': c['product_id'],
    #                        'category': c['category'], 'material': c['material'],
    #                        'text': c['description']}.
    #            then index.upsert(vectors=...)
    pass

print("Text half indexed (or skeleton placeholder; replace YOUR CODE lines).")

# Embed and upsert the 100 image products. Cohere multimodal takes images
# as base64 strings via inputs=[{'image': base64_str}].
IMAGE_BATCH = 8
image_items = [c for c in CATALOG if c["modality"] == "image"]
for batch_start in range(0, len(image_items), IMAGE_BATCH):
    batch = image_items[batch_start:batch_start + IMAGE_BATCH]
    inputs = [{"image": _png_to_base64(c["image_path"])} for c in batch]

    # YOUR CODE: call co.embed(inputs=inputs, model=COHERE_MULTIMODAL_MODEL,
    #            input_type='image') and pull .embeddings. Store as `image_vectors`.
    image_vectors = []  # replace with the embed call result

    # YOUR CODE: build upsert items as (id, vector, metadata) with
    #            metadata = {'modality': 'image', 'product_id': c['product_id'],
    #                        'category': c['category'], 'material': c['material'],
    #                        'image_path': c['image_path']}.
    #            then index.upsert(vectors=...)
    pass

print("Image half indexed (or skeleton placeholder; replace YOUR CODE lines).")

# Confirm the count.
time.sleep(3)  # Pinecone is eventually consistent on stats for ~1-2 seconds.
stats = index.describe_index_stats()
print("Index stats:", stats)

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: unified Pinecone index contains 200 vectors with mixed
# modality metadata. Validator queries Pinecone live: confirms the index
# exists, count == 200, and samples 10 random ids to inspect modality
# metadata distribution.

def _validate_unified_index(session):
    try:
        names = [ix["name"] for ix in pc.list_indexes()]
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Pinecone list_indexes failed: {exc!r}")

    if PINECONE_INDEX_NAME not in names:
        return CheckpointResult(
            "fail",
            error_reason=f"Index {PINECONE_INDEX_NAME!r} not found. Did create_index complete?",
        )

    try:
        local_index = pc.Index(PINECONE_INDEX_NAME)
        stats = local_index.describe_index_stats()
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"describe_index_stats failed: {exc!r}")

    count = stats.get("total_vector_count", 0)
    if count != TOTAL_VECTORS:
        return CheckpointResult(
            "fail",
            error_reason=f"Index has {count} vectors. Expected {TOTAL_VECTORS} (100 text + 100 image).",
        )

    # Sample 10 random product ids and fetch metadata.
    sample_ids = [f"product_{pid}" for pid in random.sample(range(TOTAL_VECTORS), 10)]
    try:
        fetched = local_index.fetch(ids=sample_ids)
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"index.fetch failed: {exc!r}")

    vectors = getattr(fetched, "vectors", None) or fetched.get("vectors", {})
    if len(vectors) < 5:
        return CheckpointResult(
            "fail",
            error_reason=(
                f"Sample fetch returned only {len(vectors)} of 10 requested ids. "
                f"Confirm upsert ids use the form 'product_<id>'."
            ),
        )

    modalities = set()
    for v in vectors.values():
        md = getattr(v, "metadata", None) or v.get("metadata", {}) or {}
        if "modality" in md:
            modalities.add(md["modality"])

    if "text" not in modalities or "image" not in modalities:
        return CheckpointResult(
            "fail",
            error_reason=(
                f"Sampled metadata modalities: {modalities or 'none'}. "
                f"Both 'text' and 'image' must appear. Confirm every upsert sets "
                f"metadata={{'modality': 'text' or 'image', 'product_id': ...}}."
            ),
        )

    return CheckpointResult("pass")


check(1, _validate_unified_index)

<details><summary>Hint 1 (nudge)</summary>

Print `index.describe_index_stats()`. If `total_vector_count` is not 200, one of the two upsert loops failed silently. If it is 200 but the checkpoint still fails, the upserts succeeded but the metadata is missing.

</details>

<details><summary>Hint 2 (direction)</summary>

Two common bugs: (1) you called `co.embed(..., model='embed-v3.0', ...)` instead of `'embed-v3.0-multimodal'`. Both return 1024-dim vectors so the upsert succeeds, but the image inputs silently fail. (2) Your upsert items did not include the `metadata` field; the validator samples 10 ids and checks both modalities are represented.

The Cohere multimodal API takes images via `inputs=[{'image': base64_str}]` plus `input_type='image'`. Text uses the legacy shape `texts=[...]` plus `input_type='search_document'`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
# Create the index
if PINECONE_INDEX_NAME not in [ix["name"] for ix in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=COHERE_EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)

index = pc.Index(PINECONE_INDEX_NAME)

# Text embeddings
resp = co.embed(
    texts=texts,
    model=COHERE_MULTIMODAL_MODEL,
    input_type="search_document",
)
text_vectors = resp.embeddings
items = [
    (
        f"product_{c['product_id']}",
        text_vectors[i],
        {
            "modality": "text",
            "product_id": c["product_id"],
            "category": c["category"],
            "material": c["material"],
            "text": c["description"],
        },
    )
    for i, c in enumerate(batch)
]
index.upsert(vectors=items)

# Image embeddings
resp = co.embed(
    inputs=inputs,
    model=COHERE_MULTIMODAL_MODEL,
    input_type="image",
)
image_vectors = resp.embeddings
items = [
    (
        f"product_{c['product_id']}",
        image_vectors[i],
        {
            "modality": "image",
            "product_id": c["product_id"],
            "category": c["category"],
            "material": c["material"],
            "image_path": c["image_path"],
        },
    )
    for i, c in enumerate(batch)
]
index.upsert(vectors=items)
```

</details>

**Wallet check.** Index built. 200 vectors (100 text, 100 image). Cohere spend: about a dime ($0.05 to $0.10 depending on how many re-runs you did). Pinecone: free tier covers the index. Your coffee cost 30x more so far.

## Task 2: Image-to-text query (image-input retrieves text descriptions)

Take each of the 5 test image ids (catalog ids 100, 110, 120, 130, 140), embed the PNG bytes with Cohere `embed-v3.0-multimodal` under `input_type='image'`, and run a unified-index query for the top-5 nearest neighbors. The cross-modal hit you are validating: at least 3 of the top-5 results have `modality='text'` AND their `product_id` is in the ground-truth relevant set for that image.

Do not pass a `filter={'modality': ...}` to `index.query`. You want results from both modalities in the same response.

In [ ]:
# NBVAL_SKIP
# Task 2: image-input queries against the unified index.
#
# Level 2 skeleton: the query embedding call and the index.query call are
# yours. The result inspection loop, the print formatting, and the
# IMAGE_QUERY_RESULTS dict (consumed by Checkpoint 2's validator) are
# already wired so a correct query call drops straight in.

IMAGE_QUERY_RESULTS = {}  # {str(query_product_id): [match, match, ...]}

for qid in TEST_IMAGE_QUERY_IDS:
    qitem = CATALOG[qid]
    qb64 = _png_to_base64(qitem["image_path"])

    # YOUR CODE: embed the image with Cohere multimodal and grab the single
    #            1024-dim vector. The call shape:
    #            co.embed(inputs=[{'image': qb64}], model=COHERE_MULTIMODAL_MODEL,
    #                     input_type='image').embeddings[0]
    query_vector = None  # replace with the embed call

    # YOUR CODE: query the unified index with top_k=5 and include_metadata=True.
    #            Do NOT pass a filter argument.
    #            results = index.query(vector=query_vector, top_k=5,
    #                                  include_metadata=True)
    results = None  # replace with the index.query call

    matches = []
    if results is not None:
        raw = getattr(results, "matches", None) or results.get("matches", [])
        for m in raw:
            md = getattr(m, "metadata", None) or m.get("metadata", {}) or {}
            score = getattr(m, "score", None) or m.get("score")
            mid = getattr(m, "id", None) or m.get("id")
            matches.append({
                "id": mid,
                "score": score,
                "modality": md.get("modality"),
                "product_id": md.get("product_id"),
                "category": md.get("category"),
                "material": md.get("material"),
                "text": md.get("text"),
                "image_path": md.get("image_path"),
            })

    IMAGE_QUERY_RESULTS[str(qid)] = matches
    print(f"Image query {qid} ({qitem['category']}): top-5 modalities = {[m['modality'] for m in matches]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: image-input queries return at least 3 relevant text
# descriptions in the top-5. Validator re-runs the queries against the
# index live (does NOT trust IMAGE_QUERY_RESULTS) so the metric is the
# cloud state, not notebook state.

def _validate_image_to_text(session):
    try:
        local_index = pc.Index(PINECONE_INDEX_NAME)
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Could not open index: {exc!r}")

    per_query_hits = {}
    for qid in TEST_IMAGE_QUERY_IDS:
        qitem = CATALOG[qid]
        try:
            qb64 = _png_to_base64(qitem["image_path"])
            resp = co.embed(
                inputs=[{"image": qb64}],
                model=COHERE_MULTIMODAL_MODEL,
                input_type="image",
            )
            qvec = resp.embeddings[0]
            results = local_index.query(vector=qvec, top_k=5, include_metadata=True)
        except Exception as exc:
            return CheckpointResult(
                "error",
                error_reason=f"Validator could not re-run image query {qid}: {exc!r}",
            )

        raw = getattr(results, "matches", None) or results.get("matches", [])
        relevant = set(RELEVANCE_LABELS["image_queries"][str(qid)])
        hits = 0
        for m in raw:
            md = getattr(m, "metadata", None) or m.get("metadata", {}) or {}
            if md.get("modality") == "text" and md.get("product_id") in relevant:
                hits += 1
        per_query_hits[qid] = hits

    failed = {qid: h for qid, h in per_query_hits.items() if h < 3}
    if failed:
        return CheckpointResult(
            "fail",
            error_reason=(
                f"Image-to-text recall below 3 for queries: {failed}. "
                f"Check that text vectors were embedded with embed-v3.0-multimodal "
                f"and that index.query is not filtered by modality."
            ),
        )

    return CheckpointResult("pass")


check(2, _validate_image_to_text)

<details><summary>Hint 1 (nudge)</summary>

Print the metadata modalities of the top-5 for one of the image queries. If they are all `image`, your retrieval is filtering. If they are mixed but the categories do not match the query image, your indexing used the wrong Cohere model.

</details>

<details><summary>Hint 2 (direction)</summary>

Cross-modal retrieval depends on two things: (1) the query embedding lives in the same space as both indexed modalities (Cohere `embed-v3.0-multimodal` puts text and image vectors in a shared space); (2) the query call does not restrict results to a single modality.

Remove any `filter={'modality': 'text'}` or `filter={'modality': 'image'}` argument from `index.query`. The unified design is the whole point.

Image inputs to Cohere multimodal use `inputs=[{'image': base64_str}]` and `input_type='image'`. If you tried passing a URL string under `texts=...`, the embedding is bogus.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
qb64 = _png_to_base64(qitem["image_path"])
query_vector = co.embed(
    inputs=[{"image": qb64}],
    model=COHERE_MULTIMODAL_MODEL,
    input_type="image",
).embeddings[0]

results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True,
)
```

</details>

**Wallet check.** 5 image-input queries. Cohere multimodal image embedding: ~$0.005 per image. Cumulative spend so far: about 12 cents. Cleanup is still cheaper than coffee.

## Task 3: Text-to-image query (text input retrieves image vectors)

Symmetric to Task 2. Embed each of the 5 test text queries with `input_type='search_query'`, run the unified-index query, and confirm at least 3 of the top-5 results have `modality='image'` AND product_ids that match the labelled-relevant set for that query.

Same anti-bug as Task 2: do not pass a `filter` argument.

In [ ]:
# NBVAL_SKIP
# Task 3: text-input queries against the unified index.

TEXT_QUERY_RESULTS = {}  # {text_query: [match, match, ...]}

for query_text in TEST_TEXT_QUERIES:
    # YOUR CODE: embed the query text with Cohere multimodal. The call shape:
    #            co.embed(texts=[query_text], model=COHERE_MULTIMODAL_MODEL,
    #                     input_type='search_query').embeddings[0]
    query_vector = None  # replace with the embed call

    # YOUR CODE: index.query(vector=query_vector, top_k=5, include_metadata=True).
    #            No filter argument.
    results = None  # replace with the index.query call

    matches = []
    if results is not None:
        raw = getattr(results, "matches", None) or results.get("matches", [])
        for m in raw:
            md = getattr(m, "metadata", None) or m.get("metadata", {}) or {}
            score = getattr(m, "score", None) or m.get("score")
            mid = getattr(m, "id", None) or m.get("id")
            matches.append({
                "id": mid,
                "score": score,
                "modality": md.get("modality"),
                "product_id": md.get("product_id"),
                "category": md.get("category"),
                "material": md.get("material"),
                "text": md.get("text"),
                "image_path": md.get("image_path"),
            })

    TEXT_QUERY_RESULTS[query_text] = matches
    print(f"Text query {query_text!r}: top-5 modalities = {[m['modality'] for m in matches]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: text-input queries return at least 3 relevant image vectors.
# Validator re-runs the queries live.

def _validate_text_to_image(session):
    try:
        local_index = pc.Index(PINECONE_INDEX_NAME)
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Could not open index: {exc!r}")

    per_query_hits = {}
    for query_text in TEST_TEXT_QUERIES:
        try:
            resp = co.embed(
                texts=[query_text],
                model=COHERE_MULTIMODAL_MODEL,
                input_type="search_query",
            )
            qvec = resp.embeddings[0]
            results = local_index.query(vector=qvec, top_k=5, include_metadata=True)
        except Exception as exc:
            return CheckpointResult(
                "error",
                error_reason=f"Validator could not re-run text query {query_text!r}: {exc!r}",
            )

        raw = getattr(results, "matches", None) or results.get("matches", [])
        relevant = set(RELEVANCE_LABELS["text_queries"][query_text])
        hits = 0
        for m in raw:
            md = getattr(m, "metadata", None) or m.get("metadata", {}) or {}
            if md.get("modality") == "image" and md.get("product_id") in relevant:
                hits += 1
        per_query_hits[query_text] = hits

    failed = {q: h for q, h in per_query_hits.items() if h < 3}
    if failed:
        return CheckpointResult(
            "fail",
            error_reason=(
                f"Text-to-image recall below 3 for queries: {failed}. "
                f"Confirm both text and image vectors were generated with "
                f"embed-v3.0-multimodal in the same API model."
            ),
        )

    return CheckpointResult("pass")


check(3, _validate_text_to_image)

<details><summary>Hint 1 (nudge)</summary>

Same shape as Checkpoint 2. Print the modalities of the top-5 for one of the text queries. If they are all `text`, retrieval is collapsing to within-modality similarity (which usually means the text and image vectors live in different sub-spaces).

</details>

<details><summary>Hint 2 (direction)</summary>

Cohere has two Embed v3 models that return 1024-dim vectors: `embed-v3.0` (text-only) and `embed-v3.0-multimodal`. Only the multimodal variant places text and image vectors in a shared space. Confirm the model string is `'embed-v3.0-multimodal'` in BOTH halves of Task 1.

Also: `input_type='search_query'` for the query, `input_type='search_document'` for indexing text, `input_type='image'` for indexing images. These are not interchangeable.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
query_vector = co.embed(
    texts=[query_text],
    model=COHERE_MULTIMODAL_MODEL,  # "embed-v3.0-multimodal"
    input_type="search_query",
).embeddings[0]

results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True,
)
```

</details>

**Wallet check.** 5 text-input queries. Cohere multimodal text embedding: fractions of a cent each. Cumulative spend: still under 15 cents. The expensive cell is next.

## Task 4: Grounded generation that cites both modalities

For each of 10 mixed-modality queries (5 image inputs + 5 text inputs), retrieve top-5 from the unified index, then call Claude Sonnet 4.6 with the retrieved chunks as `content` blocks. Text matches go in as `{'type': 'text', 'text': ...}` blocks; image matches go in as `{'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': base64_str}}` blocks.

The system prompt instructs Claude to cite every product it references with `[product_<id>]`. The checkpoint scans the trace JSON: each response must cite at least one product_id; every cited id must be in that query's retrieved top-5; across the 10 responses, at least 7 must include both a text-modality citation AND an image-modality citation.

The single biggest bug here: passing the user message as a string. The Anthropic Messages API takes `content` as a LIST of content blocks. A string-only message has zero image inputs even if the prompt mentions an image.

In [ ]:
# NBVAL_SKIP
# Task 4: grounded generation across 10 mixed-modality queries.
#
# Level 2 skeleton: the retrieve-top-5 helper, the per-query loop, and the
# trace JSON writer are provided. The construction of the user-message
# content blocks and the anthropic.messages.create call are yours.

SYSTEM_PROMPT = (
    "You are a multimodal product search assistant. The user query may be a "
    "text question or an image of a product. You will be given retrieved "
    "context as a sequence of content blocks: some are text descriptions of "
    "products and some are product images. Each block is labelled with its "
    "product_id. When you answer, cite every product you reference using the "
    "format [product_<id>]. If the retrieved context contains both text and "
    "image sources, you MUST cite at least one of each. Keep the answer to "
    "three sentences. Do not invent product ids that are not in the "
    "retrieved context."
)


def _retrieve_top5_for_image(qid):
    qb64 = _png_to_base64(CATALOG[qid]["image_path"])
    qvec = co.embed(
        inputs=[{"image": qb64}],
        model=COHERE_MULTIMODAL_MODEL,
        input_type="image",
    ).embeddings[0]
    results = index.query(vector=qvec, top_k=5, include_metadata=True)
    raw = getattr(results, "matches", None) or results.get("matches", [])
    return raw


def _retrieve_top5_for_text(query_text):
    qvec = co.embed(
        texts=[query_text],
        model=COHERE_MULTIMODAL_MODEL,
        input_type="search_query",
    ).embeddings[0]
    results = index.query(vector=qvec, top_k=5, include_metadata=True)
    raw = getattr(results, "matches", None) or results.get("matches", [])
    return raw


def _build_content_blocks(query_label, query_image_b64, matches):
    """Construct the Anthropic Messages content list for one query.

    Block order: an opening text block stating the query, then one block per
    retrieved match (text or image), then a closing instruction block.
    """
    blocks = [{"type": "text", "text": f"Query: {query_label}"}]
    if query_image_b64 is not None:
        blocks.append({
            "type": "image",
            "source": {"type": "base64", "media_type": "image/png", "data": query_image_b64},
        })
        blocks.append({"type": "text", "text": "(The block above is the query image.)"})
    blocks.append({"type": "text", "text": "Retrieved context:"})
    for m in matches:
        md = getattr(m, "metadata", None) or m.get("metadata", {}) or {}
        pid = md.get("product_id")
        modality = md.get("modality")
        if modality == "text":
            blocks.append({
                "type": "text",
                "text": f"[product_{pid}] (text) {md.get('text', '')}",
            })
        elif modality == "image":
            ipath = md.get("image_path")
            if ipath and os.path.exists(ipath):
                ib64 = _png_to_base64(ipath)
                blocks.append({"type": "text", "text": f"[product_{pid}] (image) follows:"})
                blocks.append({
                    "type": "image",
                    "source": {"type": "base64", "media_type": "image/png", "data": ib64},
                })
    blocks.append({
        "type": "text",
        "text": "Answer in three sentences. Cite each referenced product as [product_<id>].",
    })
    return blocks


ALL_QUERIES = []
for qid in TEST_IMAGE_QUERY_IDS:
    ALL_QUERIES.append({"kind": "image", "qid": qid, "label": f"product image id {qid}"})
for q in TEST_TEXT_QUERIES:
    ALL_QUERIES.append({"kind": "text", "qid": None, "label": q})

traces = []
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

for qspec in ALL_QUERIES:
    if qspec["kind"] == "image":
        matches = _retrieve_top5_for_image(qspec["qid"])
        query_image_b64 = _png_to_base64(CATALOG[qspec["qid"]]["image_path"])
    else:
        matches = _retrieve_top5_for_text(qspec["label"])
        query_image_b64 = None

    content_blocks = _build_content_blocks(qspec["label"], query_image_b64, matches)

    # YOUR CODE: call anthropic_client.messages.create with the system prompt
    #            and one user message whose `content` is the content_blocks LIST.
    #            max_tokens=400. Pull the .content[0].text out as `answer`.
    answer = ""  # replace with the messages.create call result

    retrieved_ids = []
    retrieved_modalities = []
    for m in matches:
        md = getattr(m, "metadata", None) or m.get("metadata", {}) or {}
        retrieved_ids.append(md.get("product_id"))
        retrieved_modalities.append(md.get("modality"))

    traces.append({
        "query_kind": qspec["kind"],
        "query_label": qspec["label"],
        "retrieved_ids": retrieved_ids,
        "retrieved_modalities": retrieved_modalities,
        "answer": answer,
    })

with open(TRACE_PATH, "w") as f:
    json.dump(traces, f, indent=2)

print(f"Wrote {len(traces)} grounded-generation traces to {TRACE_PATH}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: grounded generation cites both modalities across >=7 of 10.
# Validator reads the trace JSON written above. Each row needs:
#   (a) at least one [product_<id>] citation,
#   (b) every cited id is a subset of the row's retrieved_ids,
#   (c) across all rows, >=7 cite at least one text-modality match AND one
#       image-modality match.

_CITATION_RE = re.compile(r"\[product_(\d+)\]")


def _validate_grounded_generation(session):
    if not os.path.exists(TRACE_PATH):
        return CheckpointResult(
            "fail",
            error_reason=f"Trace file {TRACE_PATH} not found. Run Task 4 first.",
        )

    try:
        with open(TRACE_PATH) as f:
            rows = json.load(f)
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Trace JSON parse failed: {exc!r}")

    if len(rows) != 10:
        return CheckpointResult(
            "fail",
            error_reason=f"Expected 10 trace rows, found {len(rows)}.",
        )

    cross_modal_rows = 0
    for i, row in enumerate(rows):
        answer = row.get("answer") or ""
        cited = [int(x) for x in _CITATION_RE.findall(answer)]
        if not cited:
            return CheckpointResult(
                "fail",
                error_reason=(
                    f"Row {i} ({row.get('query_label')!r}): answer has no "
                    f"[product_<id>] citation. Tighten the system prompt or "
                    f"raise max_tokens. Sample answer: {answer[:120]!r}"
                ),
            )

        retrieved = set(row.get("retrieved_ids") or [])
        bogus = [c for c in cited if c not in retrieved]
        if bogus:
            return CheckpointResult(
                "fail",
                error_reason=(
                    f"Row {i}: model cited product_ids {bogus} that were not in "
                    f"the retrieved top-5 {sorted(retrieved)}. The system prompt "
                    f"must forbid invented ids."
                ),
            )

        ids_to_modality = {}
        for pid, modality in zip(row.get("retrieved_ids") or [], row.get("retrieved_modalities") or []):
            ids_to_modality[pid] = modality
        cited_modalities = {ids_to_modality.get(c) for c in cited}
        if "text" in cited_modalities and "image" in cited_modalities:
            cross_modal_rows += 1

    if cross_modal_rows < 7:
        return CheckpointResult(
            "fail",
            error_reason=(
                f"Only {cross_modal_rows} of 10 responses cited both a text and "
                f"an image source. The Anthropic Messages content list MUST "
                f"include image blocks (not just a text mention of the image), "
                f"and the system prompt must instruct the model to cite both."
            ),
        )

    return CheckpointResult("pass")


check(4, _validate_grounded_generation)

<details><summary>Hint 1 (nudge)</summary>

Open `multimodal_traces.json` and look at one of the answers. If the citations are all `[product_<id>]` from text-modality matches, the model never saw the retrieved images as image inputs. If the answer is empty, the API call returned no text.

</details>

<details><summary>Hint 2 (direction)</summary>

Anthropic Messages content blocks. To pass an image to Claude, the user message's `content` MUST be a list of dicts, NOT a string. Each image block is shaped:

```
{
  "type": "image",
  "source": {"type": "base64", "media_type": "image/png", "data": base64_str},
}
```

A `content` value that is a single string carries zero image inputs even if the string mentions an image. The system prompt must also explicitly instruct the model to cite both text and image sources when both appear in the retrieved context.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=400,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": content_blocks,  # the LIST returned by _build_content_blocks
        }
    ],
)
answer = resp.content[0].text
```

</details>

**Wallet check.** 10 grounded generations, each with up to ~5 image blocks plus retrieved text plus the model's answer. Roughly 15K-25K input tokens and 4K output tokens. Claude Sonnet with image input: about $1.20 to $1.80. Total session so far: about $1.50. Cleanup is next.

## Cleanup

Delete the Pinecone index, the `/content/product_images/` directory, and `/content/multimodal_traces.json`. The verification scan re-runs `list_indexes()` to confirm the index is gone and checks the local paths. A stale orphan triggers `sys.exit(1)` so the companion panel sees the dirty state.

In [ ]:
# NBVAL_SKIP
# Cleanup. Reverse-creation order. No critical-tier resources because
# Pinecone Serverless free tier has no hourly billing.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Pinecone project may still hold the lab index. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $1.50 to $2.50.** Claude Sonnet with image input was the bulk of it (~$1.20 to $1.80 for 10 grounded generations with up to 5 image blocks each). Cohere multimodal embeddings totalled about 12 cents across indexing and query embeddings. Pinecone Serverless free tier covered the index. The PNG product images and the trace JSON stayed local to the Colab kernel; cleanup removed them. If you re-ran the generation cell while debugging, add about 18 cents per re-run.

## Reflection

These are not graded. They are for you.

1. The lab indexed 100 text vectors and 100 image vectors in a single unified Pinecone index. An engineer on the team argues for splitting them into two indexes (one for text, one for image) and unioning the top-k at query time. What is the trade-off, and name one production scenario where the split-index design wins.

2. Claude Sonnet 4.6 prices image input at roughly 1100 tokens per 1.15 MP image. The product team wants to scale to 5K queries per day with 3 retrieved images each. What is the rough daily Anthropic spend, and what is one optimization you would try first to cut it WITHOUT changing the retrieval pipeline?

## Exam-Style Practice

**Question 1 (cross-modal recall asymmetry):**

A multimodal product search bot retrieves top-5 results from a unified Pinecone index containing both text descriptions and product images. The bot's hit rate on image-input queries is 78% but on text-input queries it is 41%. Which is the most likely cause?

A. The image embeddings are higher quality than the text embeddings because images contain more information.
B. The text descriptions in the catalog are short and lose information when embedded.
C. The unified index is the wrong architecture; text and images should be in separate indexes.
D. Image-input queries are inherently easier than text-input queries.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: embedding quality is a function of the model, not the modality. Cohere Embed v3 multimodal is calibrated across both modalities.
- B is correct: short text descriptions ("wooden chair") embed sparsely; the model has less signal to match against. Longer, richer descriptions would lift text-query recall significantly without touching the model.
- C is wrong: separating indexes does not change recall; it changes operational characteristics like independent scaling and separate quotas.
- D is wrong: this is a corpus characteristic, not a query-type characteristic. With rich enough text, the asymmetry inverts.

</details>

**Question 2 (image input via Anthropic Messages API):**

An engineer is feeding retrieved product images to Claude Sonnet 4.6 and the model's responses never mention any image. The retrieval pipeline returns the correct images, but the model behaves as if it sees only text. Which fix addresses the root cause?

A. Switch from Sonnet 4.6 to Opus 4.5 because Opus has stronger vision capabilities.
B. Send the image as a URL string in the user message text body so Claude can fetch it.
C. Construct the user message `content` as a list of content blocks including `{"type": "image", "source": {"type": "base64", ...}}` entries.
D. Increase `max_tokens` so the model has room to describe the image.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: model selection does not solve a payload-shape bug. Both Sonnet and Opus accept image inputs through the same content-block API.
- B is wrong: the Anthropic Messages API does not fetch URLs from user-message text. Image inputs MUST be passed as `image` content blocks with either a base64 `source` or a structured URL `source` block; a URL referenced inside a text body is invisible to the model.
- C is correct: a `content` value that is a single string carries zero image inputs. The user message must be a list of blocks, with image blocks structured exactly as shown.
- D is wrong: `max_tokens` caps output length and has no effect on whether the input contains an image.

</details>

**Question 3 (unified vs split index trade-off):**

A team is debating unified vs split vector indexes for a text-plus-image catalog. They want a single similarity query against either modality to return either modality. Which property of the embedding model determines whether a unified index can work at all?

A. The model's context window in tokens.
B. Whether the model places text and image vectors in a shared embedding space.
C. The cost per 1M tokens for text vs per 1K images.
D. The cosine vs Euclidean similarity preference.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: context window governs prompting, not embedding geometry.
- B is correct: cross-modal retrieval requires that text and image vectors live in the same space (Cohere Embed v3 multimodal is built for this; the original CLIP famously had measurable text-image misalignment). Without a shared space, a text query and an image query return results clustered within their own modality and cross-modal recall collapses.
- C is wrong: cost shape is a business concern, not a correctness condition.
- D is wrong: similarity metric is independent of whether the two modalities share a space; choosing cosine or Euclidean does not bridge separate sub-spaces.

</details>